# 🎛️ Deep Learning Studio · Go Deeper — Attention by Hand, and a Baby TabICL

**ISOM 839 · Prescriptive Analytics · Suffolk University · Prof. Hasan Arslan**

This notebook goes with the slides **"Transformers and TabICL, in Plain English."** You will do two things with your own hands:

| Part | What you do | Time |
|---|---|---|
| **1 · Attention by hand** | Compute attention in six lines: multiply, add, share out | 5 min |
| **2 · A baby TabICL** | Train a tiny transformer on 20,000 made-up tables. Then hand it tables it has **never seen** and watch it answer **without any retraining** | 12 min |
| **3 · The real thing** (optional) | Call the real TabICL on the same table | 3 min |

**Tip:** it runs on a normal Colab machine. With a GPU it is faster (Runtime → Change runtime type → T4 GPU).

| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter**, no typing needed |
| 🤔 **Bet first** | say your guess out loud *before* you run the next cell |
| ✅ **What you should see** | if your output is close to this, move on (small differences are normal) |
| ✏️ **Your turn** | change one thing and re-run |

---
## Setup

In [ ]:
# @title ▶️ Run me: setup (about 20 seconds)
import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from IPython.display import display, HTML
keras.utils.set_random_seed(839)

class Progress(keras.callbacks.Callback):
    # prints one short line every few rounds of practice
    def __init__(self, every=5):
        super().__init__(); self.every = every
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f'round {epoch + 1:3d}:  ' + '   '.join(f'{k} {v:.3f}' for k, v in logs.items()))

def card(title, rows, color='#0a66c2'):
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:760px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready! Keras', keras.__version__)

**✅ What you should see:** `✅ Ready! Keras 3.…`

---
## Part 1 · Attention by hand

The sentence: *"The animal didn't cross the street because **it** was too **tired**."* What does **"it"** mean?

In a transformer, every word gets three small lists of numbers:

| Real name | Plain English | In this cell |
|---|---|---|
| **Query** | the **question** a word asks | `question` |
| **Key** | the **name tag** a word wears | `tags` |
| **Value** | the **message** a word carries | (we reuse the tags, to keep it tiny) |

Here every tag has just two numbers: **[ how "living" , how much a "place" ]**. Real models use hundreds of numbers, and they **learn** them. We write them by hand so you can check every step in your head.

In [ ]:
# @title ▶️ Run me: attention in six lines
words    = ['animal', 'street', 'because']
tags     = np.array([[2, 0],     # animal:  very "living", not a "place"
                     [0, 2],     # street:  not living, very much a place
                     [0, 0]])    # because: neither
question = np.array([2, 0])      # "it ... was too TIRED" asks: who here is something living?

scores = tags @ question                           # 1) multiply and add  (a "dot product")
shares = np.exp(scores) / np.exp(scores).sum()     # 2) share out         (this formula is called "softmax")

for w, sc, sh in zip(words, scores, shares):
    print(f'{w:8s}  match score {sc}   share of attention {sh:6.1%}  ' + '█' * int(round(sh * 40)))
print(f'\nThe shares add up to {shares.sum():.0%}.')

**✅ What you should see:** `animal` gets about **96.5%** of the attention, `street` and `because` about **1.8%** each. The shares add up to 100%.

Check one by hand: animal's score is 2×2 + 0×0 = **4**. Street's is 2×0 + 0×2 = **0**.

**🤔 Bet first:** change the last word of the sentence to **wide** (*"…because it was too wide"*). Now "it" asks: *"who here is a place?"* That question is `[0, 2]`. Who gets the attention now?

In [ ]:
# @title ▶️ Run me: "…because it was too WIDE"
def attention(question, tags):
    scores = tags @ question
    return np.exp(scores) / np.exp(scores).sum()

shares_wide = attention(np.array([0, 2]), tags)
for w, sh in zip(words, shares_wide):
    print(f'{w:8s}  share of attention {sh:6.1%}  ' + '█' * int(round(sh * 40)))

# 3) "it" collects the messages, weighted by attention
print('\n"it" (tired) now carries:', np.round(attention(np.array([2, 0]), tags) @ tags, 2), ' -> looks like something LIVING')
print('"it" (wide)  now carries:', np.round(shares_wide @ tags, 2), ' -> looks like a PLACE')

**✅ What you should see:** `street` now gets about **96.5%**. And the last two lines show the payoff: "it" **becomes** `[1.93, 0.04]` (a living thing) in the first sentence and `[0.04, 1.93]` (a place) in the second. Same word, new meaning, because of what it paid attention to.

**That is attention.** A transformer does this for every word at once, several questions at a time ("multi-head"), and repeats it layer after layer.

**✏️ Your turn:** how strongly a word focuses depends on how big the numbers are. Run the cell below, then try your own number.

In [ ]:
# @title ✏️ Your turn: a louder or a quieter question
for strength in [0.25, 1, 2]:          # <- try 0, or 5
    sh = attention(strength * np.array([2, 0]), tags)
    print(f'question x {strength:<4}:  animal {sh[0]:6.1%}   street {sh[1]:6.1%}   because {sh[2]:6.1%}')

**✅ What you should see:** a quiet question (× 0.25) spreads attention out (animal about **58%**). A loud one (× 2) puts almost everything on one word (about **99.9%**). This is the **"how sharply it focuses"** slider from the slides.

<details><summary>🧠 <b>Check yourself:</b> what happens with strength 0? (click)</summary>

Every match score is 0, so every word gets the same share: 33.3% each. A question with no content listens to everyone equally, which is just a plain average.
</details>

---
## Part 2 · A baby TabICL: "the table is the prompt"

Normal machine learning: **one table → train one model.** New table? Train again.

TabICL works differently. It is trained **once**, on millions of made-up tables, to do one job:

> *Here are some rows **with** answers. Here are some rows **without**. Fill in the missing answers.*

After that its knobs are **frozen**. You hand it your table and it answers in one pass. This is called **in-context learning**: the learning happens by *reading the examples*, not by changing the knobs.

We will build a tiny cousin and see this with our own eyes.

| | 👶 Our baby | 🏋️ The real TabICL |
|---|---|---|
| Columns it can read | exactly 2 | hundreds (a first stage reads each column) |
| Made-up practice tables | 20,000, with 3 kinds of hidden rules | millions, far more varied |
| Practice time | a few minutes | about 20 days on three large GPUs (original paper) |
| Layers in the "learn from examples" stage | 3 | 12 |

### Step A · Make up some tables

Each practice table has **60 rows and 2 columns**, and a **hidden yes/no rule** that is different every time. 40 rows come with their answer. 20 rows have the answer hidden.

In [ ]:
# @title ▶️ Run me: the made-up-table machine
N_LABELED, N_NEW = 40, 20       # rows with answers, rows without

def make_table(rng, n):
    X = rng.normal(size=(n, 2))                     # two columns of numbers
    kind = rng.integers(3)                          # pick one of three kinds of hidden rule
    a = rng.uniform(0, 2 * np.pi)
    if kind == 0:                                   # 1) a straight-line rule
        s = X @ np.array([np.cos(a), np.sin(a)]) + 0.5 * rng.normal()
    elif kind == 1:                                 # 2) an "inside the circle" rule
        s = rng.uniform(0.8, 1.6) - np.linalg.norm(X - 0.6 * rng.normal(size=2), axis=1)
        if rng.random() < 0.5: s = -s
    else:                                           # 3) two columns that only matter together
        Z = X @ np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
        s = Z[:, 0] * Z[:, 1]
    y = (s + 0.15 * rng.normal(size=n) > 0).astype('int32')   # a little noise, like real life
    return X.astype('float32'), y

def make_many(n_tables, seed, n_labeled=N_LABELED, n_new=N_NEW):
    rng = np.random.default_rng(seed)
    XL = np.zeros((n_tables, n_labeled, 2), 'float32'); YL = np.zeros((n_tables, n_labeled), 'int32')
    XN = np.zeros((n_tables, n_new, 2), 'float32');     YN = np.zeros((n_tables, n_new, 1), 'float32')
    for t in range(n_tables):
        while True:                                 # keep only tables whose labeled rows show both answers
            X, y = make_table(rng, n_labeled + n_new)
            if 0 < y[:n_labeled].sum() < n_labeled: break
        XL[t], YL[t], XN[t], YN[t, :, 0] = X[:n_labeled], y[:n_labeled], X[n_labeled:], y[n_labeled:]
    return XL, YL, XN, YN

XL, YL, XN, YN = make_many(20_000, seed=839)
print(f'{len(XL):,} made-up tables. Each: {N_LABELED} rows with answers + {N_NEW} rows without.')

fig, axes = plt.subplots(1, 6, figsize=(16, 2.9))
for k, ax in enumerate(axes):
    ax.scatter(XL[k, :, 0], XL[k, :, 1], c=np.where(YL[k] == 1, '#d9534f', '#2e9e6b'), s=28, edgecolor='white', linewidth=0.5)
    ax.scatter(XN[k, :, 0], XN[k, :, 1], c='lightgray', marker='*', s=70, edgecolor='gray', linewidth=0.5)
    ax.set_title(f'made-up table {k + 1}', fontsize=10); ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.suptitle('red = yes,  green = no,  gray star = answer hidden.   Every table hides a different rule.', y=1.04); plt.show()

**✅ What you should see:** `20,000 made-up tables…` and six small pictures. In each one the red and green dots are split by a **different** rule: a line here, a circle there, opposite corners somewhere else.

No single rule works for all of them. To fill in the ⭐ stars, a model **has to read the red and green dots of that table**. That is the skill we are about to train.

### Step B · Build the baby

Three ideas, and you have seen all of them:

1. **Read each row** → a list of 64 numbers (a "fingerprint"). Plain `Dense` layers.
2. **Attach the answer** to every labeled row's fingerprint.
3. **Attention, twice per layer:** the labeled rows talk to each other, then each new row **listens to the labeled rows**. New rows never listen to each other.

In [ ]:
# @title ▶️ Run me: build the baby TabICL
D = 64   # size of each row's fingerprint

def build_baby():
    x_labeled = keras.Input((None, 2), name='rows_with_answers')
    y_labeled = keras.Input((None,), dtype='int32', name='their_answers')
    x_new     = keras.Input((None, 2), name='rows_without_answers')

    read_row = keras.Sequential([layers.Dense(D, activation='relu'), layers.Dense(D)], name='read_row')
    L = read_row(x_labeled) + layers.Embedding(2, D, name='attach_answer')(y_labeled)   # fingerprint + answer
    N = read_row(x_new)                                                                 # fingerprint, no answer

    for _ in range(3):                                                # 3 rounds of "talk, think"
        L = layers.LayerNormalization()(L + layers.MultiHeadAttention(4, D // 4)(L, L))   # labeled rows talk to each other
        L = layers.LayerNormalization()(L + layers.Dense(D)(layers.Dense(2 * D, activation='relu')(L)))
        N = layers.LayerNormalization()(N + layers.MultiHeadAttention(4, D // 4)(N, L))   # new rows LISTEN to labeled rows
        N = layers.LayerNormalization()(N + layers.Dense(D)(layers.Dense(2 * D, activation='relu')(N)))

    chance_of_yes = layers.Dense(1, activation='sigmoid')(N)
    return keras.Model([x_labeled, y_labeled, x_new], chance_of_yes, name='baby_tabicl')

baby = build_baby()
print(f'Baby TabICL built: {baby.count_params():,} knobs.')

**✅ What you should see:** `Baby TabICL built: 205,377 knobs.`

`MultiHeadAttention(N, L)` is Part 1 at scale: each new row asks a question, each labeled row wears a name tag, and the new row collects their messages. `4` means four questions at once.

### Step C · Practice on the made-up tables

**🤔 Bet first:** a coin flip gets 50% of the hidden answers right. How high can the baby get, on tables whose rule changes every single time?

In [ ]:
# @title ▶️ Run me: practice (about 2 minutes on a normal Colab machine, faster with a GPU)
baby.compile(keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
t0 = time.time()
baby.fit([XL, YL, XN], YN, batch_size=128, epochs=6, verbose=0, callbacks=[Progress(every=1)])
print(f'\nPractice took {time.time() - t0:.0f} seconds.')

**✅ What you should see:** accuracy starts near **0.55** and climbs to about **0.90**. (Each "round" is one pass over the 20,000 tables.)

Think about what that number means. No rule could be memorized, because every table has a different one. The baby learned **how to read examples**.

### Step D · The test: 500 brand-new tables, knobs frozen

From here on **we never train the baby again.** We compare it with two classic methods that *do* get to train on each table's 40 labeled rows:

- **Logistic regression**: the one-neuron checklist from Studio 1.
- **Nearest neighbors**: "look at the 5 most similar labeled rows and take a vote."

In [ ]:
# @title ▶️ Run me: baby (frozen) vs. two classics (trained fresh on every table)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

TL, TYL, TN, TYN = make_many(500, seed=2026)                  # 500 tables the baby has never seen
p = baby.predict([TL, TYL, TN], verbose=0)[..., 0]
acc_baby = ((p > 0.5) == TYN[..., 0]).mean()
acc_logit = np.mean([LogisticRegression().fit(TL[i], TYL[i]).score(TN[i], TYN[i, :, 0]) for i in range(500)])
acc_knn   = np.mean([KNeighborsClassifier(5).fit(TL[i], TYL[i]).score(TN[i], TYN[i, :, 0]) for i in range(500)])

card('500 new made-up tables: share of hidden answers filled in correctly', [
    ('👶 Baby TabICL (frozen, no training on these tables)', f'{acc_baby:.1%}'),
    ('Nearest neighbors (5 closest rows vote)', f'{acc_knn:.1%}'),
    ('Logistic regression (trained 500 times, once per table)', f'{acc_logit:.1%}'),
    ('Coin flip', '50.0%')], color='#f2b135')

**✅ What you should see:** the baby at about **89%**, nearest neighbors about **85%**, logistic regression about **71%**.

The baby did **zero** training on these 500 tables and still wins. Logistic regression can only draw straight lines, so it fails on the circle and corner rules. The baby has seen so many kinds of rules that it recognizes which kind it is looking at.

### Step E · See what it believes

The background color is the baby's answer for **every possible new row**: red = "yes", green = "no". Same frozen baby, three different tables.

In [ ]:
# @title ▶️ Run me: one frozen model, three different tables
def ask(X_labeled, y_labeled, X_new):
    # The labeled rows are the PROMPT. Nothing is trained here.
    out = baby.predict([np.asarray(X_labeled, 'float32')[None], np.asarray(y_labeled, 'int32')[None],
                        np.asarray(X_new, 'float32')[None]], verbose=0)
    return out[0, :, 0]

GRID = np.linspace(-3, 3, 70)
GRID_ROWS = np.array([[a, b] for b in GRID for a in GRID], 'float32')

def show_map(ax, X, y, title):
    belief = ask(X, y, GRID_ROWS).reshape(len(GRID), len(GRID))
    ax.contourf(GRID, GRID, belief, levels=np.linspace(0, 1, 11), cmap='RdYlGn_r', alpha=0.55)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(np.asarray(y) == 1, '#d9534f', '#2e9e6b'), s=34, edgecolor='white', linewidth=0.7)
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

picks = [3, 17, 7]                                    # three of the 500 new tables
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, i in zip(axes, picks):
    hit = ((ask(TL[i], TYL[i], TN[i]) > 0.5) == TYN[i, :, 0]).mean()
    show_map(ax, TL[i], TYL[i], f'new table {i}: {hit:.0%} of hidden answers right')
plt.suptitle('Dots = the 40 rows with answers (the prompt).  Background = what the frozen baby predicts everywhere.', y=1.02); plt.show()

**✅ What you should see:** three maps with three different shapes: a **straight split**, a **circle**, and **opposite corners**. In each, the red background sits behind the red dots. Each title shows the share of that table's 20 hidden answers it got right. Lines and circles are usually close to 100%. Opposite corners is the hardest of the three.

One model, never retrained, drew all three. The only thing that changed was **the table we showed it**.

**🤔 Bet first:** take the circle table and **flip every answer** (yes ↔ no). We do not retrain. What will the map do?

In [ ]:
# @title ▶️ Run me: flip the answers in the prompt (no retraining!)
i = picks[1]
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.4))
show_map(axes[0], TL[i], TYL[i], 'the table as given')
show_map(axes[1], TL[i], 1 - TYL[i], 'same rows, every answer flipped')
plt.show()
hit_flipped = ((ask(TL[i], 1 - TYL[i], TN[i]) > 0.5) == (1 - TYN[i, :, 0])).mean()
print(f'With flipped answers it still gets {hit_flipped:.0%} of the (flipped) hidden answers right.')

**✅ What you should see:** the second map is the first one with the colors swapped, and it still gets about **95%** right.

This is the whole idea of in-context learning in one picture. The "knowledge" about *this* table is not stored in the knobs. It is **read from the examples**, every time. That is why the slides say: **your table is the prompt.**

**✏️ Your turn:** more examples in the prompt, still no retraining. Does it help?

In [ ]:
# @title ✏️ Your turn: how many labeled rows does it need?
for n_labeled in [5, 10, 20, 40, 80]:            # <- try your own numbers
    A, B, C, Dd = make_many(300, seed=7, n_labeled=n_labeled)
    acc = ((baby.predict([A, B, C], verbose=0)[..., 0] > 0.5) == Dd[..., 0]).mean()
    print(f'{n_labeled:3d} rows with answers  ->  {acc:.1%} of hidden answers right   ' + '█' * int(round((acc - 0.5) * 80)))

**✅ What you should see:** accuracy climbs as the prompt grows: about **70%** with 5 labeled rows, **86%** with 20, **89%** with 40, **91%** with 80.

Same story as a chatbot: **better examples in the prompt, better answers**, with no retraining. (The baby only ever practiced with 40 labeled rows, and it still copes with other sizes.)

### Step F · A real table it has never seen

Time for real data. 178 Italian wines, each measured in a lab, from three growers. The question: **is this wine from grower 1?**

Our baby reads exactly 2 columns, so we give it 2 lab measurements at a time. It gets 40 labeled wines as the prompt and must answer for the other 138. We repeat with 20 different random prompts and average.

In [ ]:
# @title ▶️ Run me: real wines, frozen baby
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True).frame
is_grower_1 = (wine.target == 0).astype(int).values

def wine_test(col_a, col_b, n_repeats=20):
    X = wine[[col_a, col_b]].values; baby_acc, logit_acc = [], []
    for seed in range(n_repeats):
        order = np.random.default_rng(seed).permutation(len(X)); lab, new = order[:40], order[40:]
        Xs = (X - X[lab].mean(0)) / X[lab].std(0)                       # same scale, as in Studio 1
        baby_acc.append(((ask(Xs[lab], is_grower_1[lab], Xs[new]) > 0.5) == is_grower_1[new]).mean())
        logit_acc.append(LogisticRegression().fit(Xs[lab], is_grower_1[lab]).score(Xs[new], is_grower_1[new]))
    return np.mean(baby_acc), np.mean(logit_acc)

rows = []
for a, b in [('alcohol', 'proline'), ('alcohol', 'flavanoids'), ('alcohol', 'color_intensity'), ('color_intensity', 'hue')]:
    ba, lo = wine_test(a, b); rows.append((f'{a} + {b}', f'baby {ba:.1%} &nbsp;·&nbsp; logistic regression {lo:.1%}'))
card('Real wines: is it from grower 1? (40 labeled wines in the prompt)', rows, color='#7b2d8b')

order = np.random.default_rng(0).permutation(len(wine)); lab = order[:40]
X = wine[['color_intensity', 'hue']].values; Xs = (X - X[lab].mean(0)) / X[lab].std(0)
fig, ax = plt.subplots(figsize=(4.8, 4.4)); show_map(ax, Xs[lab], is_grower_1[lab], 'color_intensity + hue: what the baby believes')
ax.set_xlabel('color intensity'); ax.set_ylabel('hue'); plt.show()

**✅ What you should see:** on the first three column pairs the baby and logistic regression are **close** (within a few points of each other). On `color_intensity + hue` the baby is clearly ahead (about **87%** vs **70%**), because grower 1's wines sit in a pocket that a straight line can't cut out. The map shows the pocket.

Remember what just happened: a network that has **only ever seen made-up dots** answered questions about **real wine**, with no training on wine at all.

### Step G · The honest limit

**🤔 Bet first:** here is a shape the baby never practiced on: two interlocking moons. Will it still win?

In [ ]:
# @title ▶️ Run me: a shape it never practiced on
from sklearn.datasets import make_moons
res = {'baby': [], 'knn': [], 'logit': []}
for seed in range(20):
    X, y = make_moons(240, noise=0.2, random_state=seed); X = (X - X.mean(0)) / X.std(0)
    lab, new = slice(0, 40), slice(40, None)
    res['baby'].append(((ask(X[lab], y[lab], X[new]) > 0.5) == y[new]).mean())
    res['knn'].append(KNeighborsClassifier(5).fit(X[lab], y[lab]).score(X[new], y[new]))
    res['logit'].append(LogisticRegression().fit(X[lab], y[lab]).score(X[new], y[new]))
card('Two moons (20 different draws, 40 labeled rows each)', [
    ('Nearest neighbors', f"{np.mean(res['knn']):.1%}"), ('👶 Baby TabICL (frozen)', f"{np.mean(res['baby']):.1%}"),
    ('Logistic regression', f"{np.mean(res['logit']):.1%}")], color='#d9534f')
X, y = make_moons(240, noise=0.2, random_state=0); X = (X - X.mean(0)) / X.std(0)
fig, ax = plt.subplots(figsize=(4.8, 4.4)); show_map(ax, X[:40], y[:40], 'two moons: what the baby believes'); plt.show()

**✅ What you should see:** this time **nearest neighbors wins** (about **91%**). The baby (about **85%**) is only level with logistic regression (about **85%**). Its map shows why: it draws something close to a straight split, because lines, circles and corners are all it has ever practiced.

**This is the most important lesson of the notebook.** An in-context learner is only as broad as its practice tables. That is exactly why the real TabICL practices on **millions** of tables made from many kinds of random cause-and-effect recipes, and why its makers warn that it can slip on tables far from what it practiced (very wide ones, for example).

<details><summary>🧠 <b>Check yourself:</b> how would you make the baby better at moons, without showing it any moons at test time? (click)</summary>

Add more kinds of hidden rules to `make_table` (curved bands, several blobs, wavy lines…) and practice again. More variety in practice gives a broader skill. That is the whole recipe behind tabular foundation models.
</details>

---
## Part 3 (optional) · The real thing

The real TabICL reads **all 13 lab measurements at once**. The first run downloads the pretrained model, so it needs internet and a minute or two. A GPU helps.

If this cell fails (no internet, or the download is slow), nothing is lost: Parts 1 and 2 are the lesson.

In [ ]:
# @title ▶️ Run me (optional): the real TabICL on all 13 columns
try:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabicl'], check=True)
    from tabicl import TabICLClassifier
    X_all = wine.drop(columns='target').values
    order = np.random.default_rng(0).permutation(len(wine)); lab, new = order[:40], order[40:]
    real = TabICLClassifier(); real.fit(X_all[lab], is_grower_1[lab])           # fit only stores the 40 rows
    acc_real = (real.predict(X_all[new]) == is_grower_1[new]).mean()
    mu, sd = X_all[lab].mean(0), X_all[lab].std(0)
    acc_lr = LogisticRegression(max_iter=1000).fit((X_all[lab] - mu) / sd, is_grower_1[lab]).score((X_all[new] - mu) / sd, is_grower_1[new])
    card('Real wines, all 13 columns, 40 labeled wines', [('🏋️ Real TabICL (frozen)', f'{acc_real:.1%}'), ('Logistic regression', f'{acc_lr:.1%}')], color='#2e9e6b')
except Exception as e:
    print('Could not run the real TabICL here:', type(e).__name__, str(e)[:200])
    print('That is OK. You can try it in the Great Tabular Race notebook from Session 10.')

**✅ What you should see:** a card with two accuracies. Write them down. This is a small, easy table, so expect both to be high. For a fair fight on a bigger table, open the **Great Tabular Race** notebook from Session 10.

---
## 🎓 What you just did

| You did | The idea | The real name |
|---|---|---|
| Multiplied, added, shared out | A word decides which other words matter | **Attention** (query, key, value, softmax) |
| Practiced on 20,000 made-up tables | Learn *how to learn* from examples | **Pretraining** on synthetic data |
| Got answers on new tables with frozen knobs | The table is the prompt | **In-context learning** |
| Flipped the answers and the map flipped | The knowledge is read from the examples | (still in-context learning!) |
| Lost on the moons | Only as broad as the practice | **Out-of-distribution** data |

**For a manager:** a tabular foundation model is a strong, fast first try on a small or medium table. It is not magic. Test it on your own data against a simple baseline, exactly as you did here.

Back to the [Deep Learning Studio](https://isom-839.vercel.app/deep-learning-studio).